# Wildfire smoke PM2.5: 10-km grid to census tracts (provenance)

Kept as run. Input: Childs et al. daily 10-km smoke PM2.5 predictions
(`smokePM2pt5_predictions_daily_10km_20060101-20201231.csv`, Harvard Dataverse
doi:10.7910/DVN/DJVMTV, v1.0) and the 10-km grid shapefile from the same deposit.

The analysis uses `area_weighted_smoke_pm_corr.parquet` (written in the cell that calls
`avg_unit_smoke_pm.to_parquet("area_weighted_smoke_pm_corr.parquet")`): daily wildfire PM2.5
for each tract, averaged over intersecting grid cells weighted by intersection area, with
missing grid predictions set to 0 (no smoke). In this repository it is
`data/pm25/smoke_pm25_tract_daily.parquet`. Other cells are exploratory variants
(e.g., population weighting) that the letter does not use. Paths are those used at the time.


In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np
from pathlib import Path

## Tract map intersection with 10km_grid

In [ ]:
#unit_gdf = gpd.read_file('../shapefiles/tl_2020_06_tract/tl_2020_06_tract.shp')
unit_gdf = gpd.read_file('../../coexposure/data/WesternUSCensusTract/CensusTract2020_WesternUS.shp')

In [ ]:
unit_gdf.plot()

In [ ]:
grid_10km = gpd.read_file('10km_grid_wgs84/10km_grid_wgs84.shp')

In [ ]:
unit_gdf.crs

In [ ]:
grid_10km.crs

In [ ]:
unit_gdf = unit_gdf.to_crs('EPSG:2227')
unit_gdf['ct_area'] = unit_gdf.geometry.area

In [ ]:
grid_10km = grid_10km.to_crs(unit_gdf.crs)

In [ ]:
# make a crosswalk with intersection area with grid cells
unit_cross = gpd.overlay(unit_gdf, grid_10km, how='intersection')

In [ ]:
unit_cross = unit_cross.to_crs('EPSG:2227')

In [ ]:
unit_cross['area'] = unit_cross.geometry.area

In [ ]:
unit_cross.head()

In [ ]:
len(unit_cross.FIPS.unique())

In [ ]:
len(unit_cross)

In [ ]:
unit_cross = unit_cross[['FIPS', 'ID', 'ct_area', 'area']]

In [ ]:
unit_cross.ID.nunique() 

In [ ]:
len(unit_cross)

## Load population

In [ ]:
# csv_files = [
#     "populationDensity_10km_subgrid_1.csv",
#     "populationDensity_10km_subgrid_2.csv",
#     "populationDensity_10km_subgrid_3.csv",
#     "populationDensity_10km_subgrid_4.csv",
#     "populationDensity_10km_subgrid_5.csv",
#     "populationDensity_10km_subgrid_6.csv",
#     "populationDensity_10km_subgrid_7.csv",
#     "populationDensity_10km_subgrid_8.csv",
#     "populationDensity_10km_subgrid_9.csv",
#     "populationDensity_10km_subgrid_10.csv",
# ]

# # Read and combine CSV files
# dfs = [pd.read_csv("populationDensity_10km_subgrid/"+file) for file in csv_files]
# pop = pd.concat(dfs, ignore_index=True)

## Load Smoke PM

In [ ]:
df = pd.read_csv('smokePM2pt5_predictions_daily_10km_20060101-20201231.csv')
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

## Get weighted average on Tract

In [ ]:
# loop through work with one date at a time, concatenate the data frames from the dates
unique_dates = df['date'].unique()

In [ ]:
def weighted_mean(df, values_col, weights_col):
    temp = (df[values_col] * df[weights_col]).sum()
    if temp:
        return temp / df[weights_col].sum()
    return temp

In [ ]:
df.head()

In [ ]:
unit_cross.head()

In [ ]:
merged_df = pd.merge(df, unit_cross, left_on='grid_id_10km', right_on='ID')
merged_df.head()

In [ ]:
unit_cross.head()

In [ ]:
df.head()

In [ ]:
smoke_pm = df

In [ ]:
from tqdm import tqdm


In [ ]:
def weighted_mean(df, values_col, weights_col):
    temp = (df[values_col] * df[weights_col]).sum()
    if temp:
        return temp / df[weights_col].sum()
    return temp

In [ ]:
unit_cross.head()

In [ ]:
unit_cross['ct_area_inf'] = unit_cross.groupby('FIPS')['area'].transform('sum')
unit_cross.head()

In [ ]:
(unit_cross['ct_area_inf']-unit_cross['ct_area']).hist()

In [ ]:
avg_unit_smoke_pm = []

for date in tqdm(unique_dates, desc="Processing dates"):

    smoke_pm_date = smoke_pm[smoke_pm['date'] == date]

    smoke_pm_date_joined = unit_cross.merge(smoke_pm_date, left_on='ID', right_on='grid_id_10km', how='left')
    smoke_pm_date_joined['smokePM_pred'] = smoke_pm_date_joined['smokePM_pred'].fillna(0)

    smoke_pm_date_joined['weighted_pm'] = (smoke_pm_date_joined['smokePM_pred'] * smoke_pm_date_joined['area']) / smoke_pm_date_joined['ct_area_inf']

    smoke_pm_date_agg = smoke_pm_date_joined.groupby(['FIPS'])['weighted_pm'].sum().reset_index()

    smoke_pm_date_agg["date"] = date
    avg_unit_smoke_pm.append(smoke_pm_date_agg)

avg_unit_smoke_pm = pd.concat(avg_unit_smoke_pm, ignore_index=True)

In [ ]:
avg_unit_smoke_pm.to_parquet("area_weighted_smoke_pm_corr.parquet")

In [ ]:
avg_unit_smoke_pm[avg_unit_smoke_pm['date']==unique_dates[110]]["weighted_pm"].value_counts()

In [ ]:
avg_unit_smoke_pm[avg_unit_smoke_pm['date']==unique_dates[0]].weighted_pm.hist()

In [ ]:
# Calculating area weighted pm
merged_df['weighted_pm'] = (merged_df['smokePM_pred'] * merged_df['area']) / merged_df['ct_area']

In [ ]:
merged_df['check'] = (merged_df['area']) / merged_df['ct_area']

In [ ]:
merged_df.head()

In [ ]:
fin_df = merged_df.groupby(['date', 'FIPS'])['check'].sum()

In [ ]:
fin_df = fin_df.reset_index()

In [ ]:
fin_df[fin_df['date']==unique_dates[0]].check.hist()

In [ ]:
fin_df = merged_df.groupby(['date', 'FIPS'])['weighted_pm'].sum()

In [ ]:
len(fin_df)

In [ ]:
fin_df = fin_df.reset_index()
fin_df.head()

In [ ]:
fin_df.to_parquet("area_weighted_smoke_pm.parquet")

In [ ]:
avg_unit_smoke_pm = []
for date in unique_dates:
    smoke_pm_date = smoke_pm[smoke_pm['date'] == date]

    smoke_pm_date_joined = unit_cross.merge(smoke_pm_date, left_on='ID', right_on='grid_id_10km', how='left')
    smoke_pm_date_joined = smoke_pm_date_joined.merge(pop, on='ID', how='left')

    smoke_pm_date_joined['smokePM_pred'] = smoke_pm_date_joined['smokePM_pred'].fillna(0)
    smoke_pm_date_joined['pop'] = smoke_pm_date_joined['mean'] * smoke_pm_date_joined['area']

    smoke_pm_date_agg = smoke_pm_date_joined.groupby(['FIPS']).apply(
        lambda x: weighted_mean(x, 'smokePM_pred', 'pop')).reset_index()

    #smoke_pm_date_agg.columns = ['GEOID', 'date', 'smokePM_pred']
    smoke_pm_date_agg["date"] = date
    avg_unit_smoke_pm.append(smoke_pm_date_agg)

avg_unit_smoke_pm = pd.concat(avg_unit_smoke_pm, ignore_index=True)

In [ ]:
avg_unit_smoke_pm[avg_unit_smoke_pm[0] != 0]

In [ ]:
avg_unit_smoke_pm = avg_unit_smoke_pm.rename(columns= {0:"smoke_pm"})

In [ ]:
# Save the aggregated PM2.5 data
# avg_unit_smoke_pm.to_csv("smoke_pm_2020.csv", index=False)

In [ ]:
# Save the aggregated PM2.5 data
avg_unit_smoke_pm.to_parquet("smoke_pm_all_time_2020_map.parquet")

In [ ]:
avg_unit_smoke_pm.to_parquet("smoke_pm_all_time_2020_map_backup.parquet")

In [ ]:
len(avg_unit_smoke_pm.FIPS.unique())

In [ ]:
avg_unit_smoke_pm.head()

In [ ]:
avg_unit_smoke_pm.date.min()

In [ ]:
len(avg_unit_smoke_pm)

## Test & compare

In [ ]:
!pwd

In [ ]:
smoke_df = pd.read_parquet("../../outputs/smoke_pm25_predicted_with_fips.parquet")

In [ ]:
smoke_df.head()

In [ ]:
smoke_df.dtypes

In [ ]:
avg_unit_smoke_pm['GEOID']=avg_unit_smoke_pm['GEOID'].astype(int)

In [ ]:
fin = avg_unit_smoke_pm.merge(smoke_df, left_on=['GEOID','date'], right_on=['GEOID','time'])

In [ ]:
fin["diff"]=fin["smokePM_pred"]-fin["smoke_pm"]

In [ ]:
fin["diff"]=np.abs(fin["diff"]).round()

In [ ]:
fin["diff"].describe()